# Packing must preserve the prediction problem

Chapter 6 · Day 9

We can investigate padding efficiency without another Spark training run. Construct a document-isolated causal mask and deliberately replace it with a global causal mask. This is a CPU mechanism control, not a throughput benchmark.

In [ ]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/dongxi_llms').exists())
sys.path.insert(0, str(ROOT / 'src'))
import torch
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(909)
plt.rcParams.update({'figure.figsize': (8, 3.6), 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': False})

Prediction: if two stories are concatenated with an EOS separator, does a normal causal mask prevent story2 from reading story1? Reference: it prevents the future, but does not identify document boundaries.

In [ ]:
from dongxi_llms.course_foundations import visible_sources
document_ids = [0, 0, 0, 1, 1, 2, 2, 2]
isolated = visible_sources(document_ids)
ordinary = torch.ones(8,8,dtype=torch.bool).tril()
q = torch.arange(8*4, dtype=torch.float64).reshape(8,4) / 30
v = torch.flip(q, [0])
def output(mask):
    return (q @ q.T / 2).masked_fill(~mask, -torch.inf).softmax(-1) @ v
good, bad = output(isolated), output(ordinary)
assert isolated[3,0].item() is False and ordinary[3,0].item() is True
assert not torch.allclose(good[3], bad[3])
print('First token of second document, isolated:', good[3].tolist())
print('First token of second document, cross-document:', bad[3].tolist())

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(9,3.6))
for ax, mask, title in zip(axes,[isolated,ordinary],['Document-isolated causal','Global causal: leaks earlier documents']):
    ax.imshow(mask, cmap='Greys', vmin=0, vmax=1)
    ax.set(xlabel='Source position', ylabel='Query position', title=title)
fig.tight_layout()
plt.show()

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![02 masking and padding control: reference plot 01](../figures/chapter-06/day-09-02_masking_and_padding_control-01.png)

This fixed image comes from the CPU reference or the explicitly identified saved evidence; changing inputs does not update the preview automatically.

Reference explanation: permitted sources are the intersection of same-document membership and past/self positions. The two masks yield different states at the second document's first position. Efficient packing must not silently change document visibility. Position IDs and target boundaries also require declared treatment.

Intervention: replace v for document0 while holding all other values fixed. Reference: with the isolated mask, outputs for documents1 and2 are invariant. With ordinary causal attention they can change. Check the claim with actual tensors before interpreting any training-speed comparison.

In [ ]:
altered = v.clone(); altered[:3] += 100
def out_for(values, mask):
    return (q @ q.T/2).masked_fill(~mask,-torch.inf).softmax(-1) @ values
assert torch.allclose(out_for(altered,isolated)[3:],good[3:])
assert not torch.allclose(out_for(altered,ordinary)[3:],bad[3:])
print('Cross-document intervention invariants passed.')

This verified failure control establishes information-flow equivalence conditions. It does not implement a production packed attention kernel or establish a Spark speedup. A future length-aware comparison must additionally check data order, target normalization and context length.